# Tome C2, Chapter 10: Number-Theoretic Algorithms (Python companion)

Exercises first, solutions at the end. We implement the chapter from the bottom up: binary powering and the extended Euclidean algorithm with their cofactor bounds, Chinese remaindering and an exact determinant computed modulo many primes, the square root algorithms of Tonelli–Shanks and Cipolla, Carmichael numbers by Korselt's criterion, the Miller–Rabin and Solovay–Strassen tests with exact liar counts, the AKS test with polynomials modulo $(X^r-1,n)$, and three factoring methods: Pollard's rho, Pollard's $p-1$, and a small congruence-of-squares factoriser in the spirit of Dixon and the quadratic sieve.

*Before each solution, predict what the output will look like.*

In [ ]:
import random, math, time
from math import gcd, isqrt
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
import sympy
random.seed(10); rng = np.random.default_rng(10)

## Exercises

**P10.1 (★) Powers, inverses, remainders.** Implement left-to-right binary powering modulo $n$ and count its multiplications (Proposition 10.1); implement the extended Euclidean algorithm and check the cofactor bound $|u|\le b$ on many random pairs; implement Chinese remaindering (Proposition 10.2). Then compute the determinant of a random $40\times40$ integer matrix modulo enough $30$-bit primes, reconstruct it with the Hadamard bound (Exercise 10.2), and compare with an exact fraction-free elimination.

**P10.2 (★★) Square roots modulo $p$.** Implement Tonelli–Shanks (Theorem 10.3) and Cipolla (Theorem 10.4), check them on every square modulo a few small primes and on random squares modulo large ones, and compare their costs on a prime $p$ for which $p-1$ is divisible by a huge power of $2$. Check the count of suitable $t$ in Cipolla's algorithm.

**P10.3 (★) Carmichael numbers.** List the Carmichael numbers below $10^6$ with Korselt's criterion (Theorem 10.7), check the list against the definition below $2\cdot10^4$, and check Lemma 10.8.

**P10.4 (★★) Liars.** For every odd composite $n<3000$, count Fermat, Euler and strong liars. Check Theorem 10.10, Theorem 10.11 and the inclusions between the three sets; find the composites where the proportion of strong liars is largest. Then build a deterministic Miller–Rabin test for $64$-bit integers with the first twelve prime bases and test it against `sympy.isprime`, including on known strong pseudoprimes.

**P10.5 (★★★) AKS.** Implement the AKS algorithm exactly as in the chapter, with polynomials modulo $(X^r-1,n)$ stored as `numpy` arrays. Check it against `sympy.isprime` for all $n<1000$ and on larger numbers, print $r$ and $\ell$, and measure how the running time grows with $\log n$. Check the example of Exercise 10.7.

**P10.6 (★★) Rho.** Measure the tail and cycle lengths $\mu+\lambda$ of $x\mapsto x^2+1$ modulo many random primes, compare with the birthday bound $\sqrt{\pi p/2}$ of Proposition 10.18, then factor numbers with Pollard's rho (Floyd and Brent versions) and with the $p-1$ method.

**P10.7 (★★★) Congruence of squares.** Factor a semiprime of about $18$ digits by collecting values $x^2-n$ that factor over small primes, finding a dependency of their exponent vectors over $\mathbb F_2$, and applying Proposition 10.20.

## Solutions

### P10.1 Powers, inverses, remainders

In [ ]:
def powmod(a, e, n):
    # left-to-right binary powering; returns (a^e mod n, number of multiplications)
    y, mults = 1, 0
    for bit in bin(e)[2:]:
        y = y * y % n; mults += 1
        if bit == '1':
            y = y * a % n; mults += 1
    return y, mults - 2   # the first squaring and multiplication act on 1 and are free

def egcd(a, b):
    # extended Euclid tracking only s (cofactor of a), as in the proof of Proposition 10.1
    r0, r1, s0, s1 = a, b, 1, 0
    while r1:
        q = r0 // r1
        r0, r1 = r1, r0 - q * r1
        s0, s1 = s1, s0 - q * s1
    g, u = r0, s0
    v = (g - u * a) // b
    return g, u, v

def crt(residues, moduli):
    M = math.prod(moduli); x = 0
    for c, m in zip(residues, moduli):
        Mi = M // m
        _, _, v = egcd(m, Mi % m)      # v * (Mi mod m) = 1 mod m
        x += c * Mi * v
    return x % M

for _ in range(5):
    n = random.getrandbits(256) | 1; a = random.randrange(n); e = random.getrandbits(256)
    y, m = powmod(a, e, n)
    print(f"e has {e.bit_length()} bits, {bin(e).count('1')} ones: {m} multiplications (bound {2*(e.bit_length()-1)}), correct: {y == pow(a, e, n)}")
worst = 0
for _ in range(20000):
    a = random.getrandbits(random.randint(2, 200)) + 2; b = random.randrange(1, a)
    g, u, v = egcd(a, b)
    assert u * a + v * b == g == gcd(a, b) and abs(u) <= b and abs(v) <= a
print("extended Euclid: 20000 random pairs, ua+vb=gcd, |u|<=b and |v|<=a in every case")
moduli = [7, 11, 13, 16, 9]; x = 31415 % math.prod(moduli)
print("CRT:", crt([x % m for m in moduli], moduli), "expected", x)

The binary method uses at most $2\lfloor\log_2e\rfloor$ multiplications, and about $1.5\log_2e$ on average since half the bits are ones. Now the determinant. Each row of a $40\times40$ matrix with entries in $[-100,100]$ has norm at most $100\sqrt{40}$, so Hadamard gives $|\det A|\le(100\sqrt{40})^{40}\approx10^{112}$; with the actual row norms of a random matrix the bound is about $10^{102}$. The reference is Bareiss' fraction-free elimination, which keeps all intermediate numbers integral.

In [ ]:
def det_mod_p(A, p):
    A = [[x % p for x in row] for row in A]; n = len(A); d = 1
    for c in range(n):
        piv = next((r for r in range(c, n) if A[r][c]), None)
        if piv is None: return 0
        if piv != c: A[c], A[piv] = A[piv], A[c]; d = -d
        d = d * A[c][c] % p; inv = pow(A[c][c], -1, p)
        for r in range(c + 1, n):
            f = A[r][c] * inv % p
            if f:
                A[r] = [(x - f * y) % p for x, y in zip(A[r], A[c])]
    return d % p

def det_bareiss(A):
    A = [row[:] for row in A]; n = len(A); sign = 1; prev = 1
    for k in range(n - 1):
        if A[k][k] == 0:
            sw = next((r for r in range(k + 1, n) if A[r][k]), None)
            if sw is None: return 0
            A[k], A[sw] = A[sw], A[k]; sign = -sign
        for i in range(k + 1, n):
            for j in range(k + 1, n):
                A[i][j] = (A[i][j] * A[k][k] - A[i][k] * A[k][j]) // prev
        prev = A[k][k]
    return sign * A[n - 1][n - 1]

N = 40
A = [[random.randint(-100, 100) for _ in range(N)] for _ in range(N)]
H = math.prod(math.isqrt(sum(x * x for x in row)) + 1 for row in A)   # Hadamard bound, rounded up
primes, P, p = [], 1, 2**30
while P <= 2 * H:
    p = sympy.prevprime(p); primes.append(p); P *= p
t0 = time.perf_counter()
res = [det_mod_p(A, p) for p in primes]
D = crt(res, primes); D = D - P if D > P // 2 else D
t1 = time.perf_counter() - t0
t0 = time.perf_counter(); D2 = det_bareiss(A); t2 = time.perf_counter() - t0
print(f"Hadamard bound ~ 10^{len(str(H))-1}, {len(primes)} primes of 30 bits")
print(f"det A = {D}")
print(f"modular: {t1:.3f} s, Bareiss: {t2:.3f} s, agree: {D == D2}")

The determinant has only a few digits fewer than the Hadamard bound (random rows are nearly orthogonal in high dimension, so the bound is not far off). Both methods are exact. Here Bareiss is faster: its numbers stay around a hundred digits, and Python's big integers handle that well, while the modular method pays for twelve eliminations written in pure Python. The strengths of the modular method lie elsewhere: it never handles a number above $2^{60}$ before the final reconstruction, so in a compiled language each elimination runs on machine words, the twelve computations are independent and parallelise perfectly, and the approach extends to settings, such as matrices of polynomials, where intermediate numbers swell far worse. This is how computer algebra systems compute determinants of large integer matrices.

### P10.2 Square roots modulo $p$

In [ ]:
def legendre(a, p):
    t = pow(a, (p - 1) // 2, p)
    return -1 if t == p - 1 else t

def nonsquare(p):
    while True:
        z = random.randrange(2, p)
        if legendre(z, p) == -1: return z

def tonelli(a, p, z=None):
    a %= p
    if a == 0: return 0, 0
    s, q = 0, p - 1
    while q % 2 == 0: q //= 2; s += 1
    if z is None: z = nonsquare(p)
    m, c, x, b = s, pow(z, q, p), pow(a, (q + 1) // 2, p), pow(a, q, p)
    mults = 3 * q.bit_length()
    while b != 1:
        i, t = 0, b
        while t != 1: t = t * t % p; i += 1; mults += 1
        d = c
        for _ in range(m - i - 1): d = d * d % p; mults += 1
        m, c, x, b = i, d * d % p, x * d % p, b * d * d % p; mults += 4
    return x, mults

def cipolla(a, p):
    a %= p
    if a == 0: return 0, 0
    trials = 0
    while True:
        t = random.randrange(p); trials += 1
        w = (t * t - a) % p
        if w and legendre(w, p) == -1: break
    def mul(u, v):   # (u0 + u1 w)(v0 + v1 w) in F_p[omega], omega^2 = w
        return ((u[0] * v[0] + u[1] * v[1] * w) % p, (u[0] * v[1] + u[1] * v[0]) % p)
    e, r, base, mults = (p + 1) // 2, (1, 0), (t, 1), 0
    while e:
        if e & 1: r = mul(r, base); mults += 4
        base = mul(base, base); mults += 4; e >>= 1
    assert r[1] == 0
    return r[0], mults

for p in (3, 5, 13, 17, 97, 257, 7681):
    sq = {x * x % p for x in range(1, p)}
    ok = all(tonelli(a, p)[0] ** 2 % p == a and cipolla(a, p)[0] ** 2 % p == a for a in sq)
    print(f"p={p:5d}: all {len(sq)} nonzero squares rooted correctly by both: {ok}")
for bits in (64, 256, 1024):
    p = sympy.randprime(2**(bits - 1), 2**bits); a = pow(random.randrange(2, p), 2, p)
    print(f"{bits}-bit p: Tonelli ok {tonelli(a, p)[0]**2 % p == a}, Cipolla ok {cipolla(a, p)[0]**2 % p == a}")

In [ ]:
# a prime with p - 1 divisible by a huge power of two: p = k * 2^s + 1
s = 500
k = next(k for k in range(1, 10**6, 2) if sympy.isprime(k * 2**s + 1))
p = k * 2**s + 1
print(f"p = {k} * 2^{s} + 1, a {p.bit_length()}-bit prime")
z = nonsquare(p)
T, Cm = [], []
for _ in range(5):
    a = pow(random.randrange(2, p), 2, p)
    x1, m1 = tonelli(a, p, z); x2, m2 = cipolla(a, p)
    assert x1 * x1 % p == a and x2 * x2 % p == a
    T.append(m1); Cm.append(m2)
print("Tonelli-Shanks multiplications:", T)
print("Cipolla multiplications in F_p:", Cm)
# count of suitable t (Theorem 10.4) for a few p, a
for p in (11, 101, 1009):
    for a in sorted({x * x % p for x in range(1, p)})[:3]:
        cnt = sum(legendre((t * t - a) % p, p) == -1 for t in range(p))
        print(f"p={p}, a={a}: {cnt} suitable t, (p-1)/2 = {(p-1)//2}")

When $s$ is large, Tonelli–Shanks pays about $s^2/4$ squarings on average in its inner loops (here some $65\,000$ for $s=500$), while Cipolla's cost depends only on $\log p$: four products in $\mathbb F_p$ per multiplication in $\mathbb F_{p^2}$, at most $8\log_2p$ in all, and here about half that because the exponent $(p+1)/2$ has few binary ones. For the usual primes, with small $s$, Tonelli–Shanks is the faster of the two. The count of suitable $t$ is exactly $\frac{p-1}2$, as proved.

### P10.3 Carmichael numbers

In [ ]:
LIM = 10**6
spf = list(range(LIM))
for i in range(2, isqrt(LIM) + 1):
    if spf[i] == i:
        for j in range(i * i, LIM, i):
            if spf[j] == j: spf[j] = i
def factor_small(n):
    f = []
    while n > 1:
        p = spf[n]; e = 0
        while n % p == 0: n //= p; e += 1
        f.append((p, e))
    return f
carm = []
for n in range(3, LIM, 2):
    f = factor_small(n)
    if len(f) >= 2 and all(e == 1 and (n - 1) % (p - 1) == 0 for p, e in f):
        carm.append(n)
print(len(carm), "Carmichael numbers below 10^6; the first ten:", carm[:10])
brute = [n for n in range(3, 20000, 2) if not sympy.isprime(n) and all(pow(a, n - 1, n) == 1 for a in range(2, n) if gcd(a, n) == 1)]
print("definition below 2*10^4:", brute, " agrees with Korselt:", brute == [c for c in carm if c < 20000])
print("fewest prime factors among them:", min(len(factor_small(c)) for c in carm))

There are $43$ Carmichael numbers below $10^6$, and every one has at least three prime factors, as Lemma 10.8 says ($561=3\cdot11\cdot17$, $1105=5\cdot13\cdot17$, $1729=7\cdot13\cdot19$). The brute-force check runs over all units, which is what makes Korselt's criterion valuable: it needs only the factorisation.

### P10.4 Liars

In [ ]:
def jacobi(a, n):
    # Jacobi symbol by reciprocity (Tome 1, Proposition 6.8), n odd positive
    a %= n; result = 1
    while a:
        while a % 2 == 0:
            a //= 2
            if n % 8 in (3, 5): result = -result
        a, n = n, a
        if a % 4 == 3 and n % 4 == 3: result = -result
        a %= n
    return result if n == 1 else 0

def is_strong_liar(a, n):
    d, s = n - 1, 0
    while d % 2 == 0: d //= 2; s += 1
    y = pow(a, d, n)
    if y == 1 or y == n - 1: return True
    for _ in range(s - 1):
        y = y * y % n
        if y == n - 1: return True
    return False

stats, inclusions_ok = [], True
for n in range(9, 3000, 2):
    if sympy.isprime(n): continue
    units = [a for a in range(1, n) if gcd(a, n) == 1]
    F = {a for a in units if pow(a, n - 1, n) == 1}
    E = {a for a in units if jacobi(a, n) % n == pow(a, (n - 1) // 2, n)}
    S = {a for a in units if is_strong_liar(a, n)}
    inclusions_ok &= S <= E <= F
    stats.append((n, len(units), len(F), len(E), len(S)))
print("strong <= Euler <= Fermat for every n:", inclusions_ok)
print("max strong/(n-1):", max(s / (n - 1) for n, _, _, _, s in stats), " (Theorem 10.10: <= 1/4)")
print("max Euler/phi(n):", max(e / u for n, u, _, e, _ in stats), " (Theorem 10.11: <= 1/2)")
top = sorted(stats, key=lambda t: -t[4] / (t[0] - 1))[:6]
for n, u, f, e, s in top:
    print(f"n={n:5d} = {sympy.factorint(n)}: strong {s:4d}/{n-1} = {s/(n-1):.4f}, Euler {e}, Fermat {f}")

The inclusions $\text{strong}\subseteq\text{Euler}\subseteq\text{Fermat}$ hold everywhere (the second because an Euler liar satisfies $a^{n-1}=(\pm1)^2$, the first by the theorem of Pomerance, Selfridge and Wagstaff quoted in the chapter). The worst cases for Miller–Rabin are $9$, with exactly $\frac14$, and numbers $n=(2k+1)(4k+1)$ with both factors prime such as $703=19\cdot37$ and $1891=31\cdot61$, close to $\frac14$: for those, $\nu$ is as large as possible and $G$ has index exactly $4$.

In [ ]:
# a picture of the proportions
ns = [t[0] for t in stats]
plt.figure(figsize=(9, 3.6))
plt.scatter(ns, [t[2] / (t[0] - 1) for t in stats], s=3, label="Fermat")
plt.scatter(ns, [t[3] / (t[0] - 1) for t in stats], s=3, label="Euler")
plt.scatter(ns, [t[4] / (t[0] - 1) for t in stats], s=3, label="strong")
plt.axhline(0.25, ls="--", c="k", lw=0.8); plt.axhline(0.5, ls=":", c="k", lw=0.8)
plt.xlabel("odd composite n"); plt.ylabel("proportion of liars"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
BASES12 = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37)
def mr64(n):
    if n < 2: return False
    for p in BASES12:
        if n % p == 0: return n == p
    return all(is_strong_liar(a, n) for a in BASES12)

# known strong pseudoprimes: to base 2 (2047), to bases 2 and 3 (1373653), to bases 2,3,5,7 (3215031751),
# and the smallest strong pseudoprime to the first eleven prime bases (3825123056546413051)
for n in (2047, 1373653, 3215031751, 3825123056546413051):
    fooled = [a for a in BASES12 if is_strong_liar(a, n)]
    print(f"{n} = {sympy.factorint(n)}: fooled by bases {fooled}, mr64 says prime: {mr64(n)}")
bad = 0; t0 = time.perf_counter()
for _ in range(200000):
    n = random.getrandbits(64) | 1
    bad += mr64(n) != sympy.isprime(n)
print(f"200000 random odd 64-bit numbers: {bad} disagreements with sympy ({time.perf_counter() - t0:.1f} s)")

The number $3825123056546413051=149491\cdot747451\cdot34233211$ fools the first eleven prime bases and is caught only by $37$: this is why twelve bases are needed. No disagreement appears on random inputs, as the theorem of Sorenson and Webster guarantees for all $n<3.18\times10^{23}$.

### P10.5 AKS

In [ ]:
def is_perfect_power(n):
    for b in range(2, n.bit_length() + 1):
        a = round(n ** (1.0 / b))
        for c in (a - 1, a, a + 1):
            if c >= 2 and c ** b == n: return True
    return False

def mult_order(n, r, bound):
    # least k <= bound with n^k = 1 mod r, or None if there is none
    x = 1
    for k in range(1, bound + 1):
        x = x * n % r
        if x == 1: return k
    return None

def polymulmod(f, g, r, n):
    h = np.convolve(f, g)              # coefficients below r * n^2 < 2^63 for n < 3 * 10^6
    out = h[:r].copy(); out[:len(h) - r] += h[r:]
    return out % n

def polypowmod(f, e, r, n):
    result = np.zeros(r, dtype=np.int64); result[0] = 1
    for bit in bin(e)[2:]:
        result = polymulmod(result, result, r, n)
        if bit == '1': result = polymulmod(result, f, r, n)
    return result

def aks(n, info=False):
    assert n < 3 * 10**6
    if n < 2: return False
    if is_perfect_power(n): return False
    L = math.log2(n)
    r = 2
    while True:
        if gcd(r, n) == 1 and mult_order(n % r, r, int(L * L)) is None: break
        r += 1
    for a in range(2, min(r, n - 1) + 1):
        if 1 < gcd(a, n) < n: return False
    if n <= r: return True
    ell = int(math.sqrt(sympy.totient(r)) * L)
    if info: print(f"  n={n}: r={r}, ell={ell}")
    for a in range(1, ell + 1):
        f = np.zeros(r, dtype=np.int64); f[0] = a; f[1] = 1
        lhs = polypowmod(f, n, r, n)
        rhs = np.zeros(r, dtype=np.int64); rhs[n % r] = 1; rhs[0] = (rhs[0] + a) % n
        if not np.array_equal(lhs, rhs): return False
    return True

t0 = time.perf_counter()
print("AKS agrees with sympy for all n < 1000:", all(aks(n) == sympy.isprime(n) for n in range(1000)), f"({time.perf_counter() - t0:.1f} s)")

In [ ]:
rows = []
for digits in (3, 4, 5, 6):
    p = sympy.prevprime(10**digits)
    t0 = time.perf_counter(); ok = aks(p, info=True); t = time.perf_counter() - t0
    rows.append((math.log2(p), t)); print(f"  prime {p}: AKS says {ok}, {t:.2f} s")
for n in (1009 * 1013, 561 * 1105, 997 ** 2, 1299709 * 2 + 1):
    print(f"n={n}: AKS {aks(n)}, sympy {sympy.isprime(n)}")
L_, T_ = np.array(rows).T
slope = np.polyfit(np.log(L_), np.log(T_), 1)[0]
print(f"empirical exponent of log n in the running time on these primes: {slope:.1f}")

In [ ]:
# Exercise 10.7: 341 passes the identity with r = 2 and a = 1
f = np.array([1, 1], dtype=np.int64)
print("(X+1)^341 mod (X^2-1, 341):", polypowmod(f, 341, 2, 341), " X^341 + 1 reduces to [1 1]; 2^340 mod 341 =", pow(2, 340, 341))

For the smallest $n$ step 4 answers at once, since $r$ exceeds $n$. Beyond that, the polynomial identities dominate: $r$ is a few hundred and $\ell$ about as large, so each prime costs hundreds of polynomial powers. The empirical exponent over this small range is far from the asymptotic $10.5$ of Proposition 10.16 (which is a worst-case bound, and $r$ is in practice close to $\log^2n$, not $\log^5n$), but the contrast with Miller–Rabin, which answers for a $64$-bit number in microseconds, is the real lesson: AKS settled a question of principle, not of practice.

### P10.6 Rho

In [ ]:
def rho_lengths(p, c=1, x0=2):
    # exact tail mu and cycle lambda of x -> x^2 + c mod p (Brent's method for lambda, then mu)
    f = lambda x: (x * x + c) % p
    power = lam = 1; tort = x0; hare = f(x0)
    while tort != hare:
        if power == lam: tort = hare; power *= 2; lam = 0
        hare = f(hare); lam += 1
    tort = hare = x0
    for _ in range(lam): hare = f(hare)
    mu = 0
    while tort != hare: tort = f(tort); hare = f(hare); mu += 1
    return mu, lam

ratios, ps = [], []
for _ in range(400):
    p = sympy.randprime(10**5, 10**8)
    mu, lam = rho_lengths(p, c=random.randrange(1, p - 2), x0=random.randrange(p))
    ratios.append((mu + lam) / math.sqrt(math.pi * p / 2)); ps.append(p)
print(f"mean of (mu+lambda)/sqrt(pi p/2) over 400 random primes and random c, x0: {np.mean(ratios):.3f}")
plt.figure(figsize=(6, 3.2)); plt.hist(ratios, bins=30); plt.xlabel("(mu+lambda)/sqrt(pi p/2)"); plt.tight_layout(); plt.show()

For a truly random map the mean ratio would be close to $1$ (the birthday bound is an upper bound, and asymptotically sharp). For $x\mapsto x^2+c$ it comes out close to $1$ as well, although this map is far from random: it is two-to-one, and quadratic. The $\sqrt p$ law, which is all the rho method needs, is confirmed experimentally, and it remains unproved.

In [ ]:
def rho_floyd(n, c=1):
    f = lambda x: (x * x + c) % n
    x = y = 2; g = 1; steps = 0
    while g == 1:
        x = f(x); y = f(f(y)); g = gcd(y - x, n); steps += 1
    return (g if g != n else None), steps

def rho_brent(n, c=1):
    f = lambda x: (x * x + c) % n
    y, r, g, steps = 2, 1, 1, 0
    while g == 1:
        x = y
        for _ in range(r): y = f(y)
        k = 0
        while k < r and g == 1:
            y = f(y); g = gcd(y - x, n); k += 1; steps += 1
        r *= 2
    return (g if g != n else None), steps

for bits in (40, 50, 60):
    p = sympy.randprime(2**(bits // 2 - 1), 2**(bits // 2)); q = sympy.randprime(2**(bits // 2), 2**(bits // 2 + 1))
    n = p * q
    t0 = time.perf_counter(); g1, s1 = rho_floyd(n); t1 = time.perf_counter() - t0
    t0 = time.perf_counter(); g2, s2 = rho_brent(n); t2 = time.perf_counter() - t0
    print(f"n={n} ({bits} bits): Floyd found {g1} in {s1} steps ({t1:.3f} s), Brent found {g2} in {s2} steps ({t2:.3f} s); sqrt(p)={isqrt(min(p, q))}")
F6 = 2**64 + 1
t0 = time.perf_counter(); g, s = rho_brent(F6)
print(f"F6 = 2^64+1: factor {g} after {s} steps, {time.perf_counter() - t0:.3f} s")

In [ ]:
def pminus1(n, B, a=2):
    for q in sympy.primerange(2, B + 1):
        qe = q
        while qe * q <= B: qe *= q
        a = pow(a, qe, n)
    g = gcd(a - 1, n)
    return g if 1 < g < n else None

print("299 with B=4:", pminus1(299, 4))
# a p - 1 friendly number: p - 1 is 100-powersmooth, q - 1 is not
small = list(sympy.primerange(3, 100))
while True:
    p = 2 * math.prod(random.sample(small, 12)) + 1
    if sympy.isprime(p): break
q = sympy.nextprime(10**17)
while max(sympy.factorint(q - 1)) < 10**6: q = sympy.nextprime(q)
n = p * q
print(f"p-1 = {sympy.factorint(p - 1)}, largest prime of q-1 = {max(sympy.factorint(q - 1))}")
t0 = time.perf_counter(); print(f"p-1 method with B=100 on a {n.bit_length()}-bit n: found {pminus1(n, 100)} (p = {p}) in {time.perf_counter() - t0:.4f} s")

The number of steps is close to $\sqrt p$ for the smaller factor $p$, as the heuristic predicts. Brent's version takes about as many comparisons as Floyd's here, but it evaluates $f$ once per comparison instead of three times. The sixth Fermat number $2^{64}+1$, factored by Fortuné Landry in 1880 by hand, falls instantly. The eighth, $2^{256}+1$, factored by Brent and Pollard with rho, has a smallest factor of $16$ digits and would need some $\sqrt{1.2\times10^{15}}\approx3.5\times10^7$ steps, a minute or so of pure Python; try it if you are patient. The $p-1$ method factors the $p-1$ example instantly with $B=100$ because $p-1$ is a product of distinct primes below $100$, although $p$ has about twenty digits; the other factor $q$, whose $q-1$ has a large prime factor, is out of its reach.

### P10.7 Congruence of squares

In [ ]:
def factor_squares(n, B, extra=10, verbose=True):
    base = [p for p in sympy.primerange(2, B) if p == 2 or legendre(n % p, p) == 1]
    K = len(base); rels = []; x = isqrt(n) + 1; tried = 0
    while len(rels) < K + extra:
        Q = x * x - n; e = []; m = Q; tried += 1
        for p in base:
            c = 0
            while m % p == 0: m //= p; c += 1
            e.append(c)
        if m == 1: rels.append((x, e))
        x += 1
    if verbose: print(f"factor base: {K} primes below {B}; {len(rels)} smooth values among {tried} candidates")
    # Gaussian elimination over F_2, rows as bitmasks, tracking which relations are combined
    rows = [(sum((c & 1) << j for j, c in enumerate(e)), 1 << i) for i, (_, e) in enumerate(rels)]
    pivots = {}
    for vec, comb in rows:
        while vec:
            j = vec.bit_length() - 1
            if j not in pivots: pivots[j] = (vec, comb); break
            pv, pc = pivots[j]; vec ^= pv; comb ^= pc
        if vec == 0:     # a dependency: the product of these x^2 - n is a square
            idx = [i for i in range(len(rels)) if comb >> i & 1]
            X = math.prod(rels[i][0] for i in idx) % n
            tot = [sum(rels[i][1][j] for i in idx) for j in range(K)]
            Y = math.prod(pow(p, t // 2, n) for p, t in zip(base, tot)) % n
            assert (X * X - Y * Y) % n == 0
            g = gcd(X - Y, n)
            if 1 < g < n: return g, len(idx)
    return None, 0

for digits, B in ((12, 400), (14, 1200), (18, 3000)):
    p = sympy.randprime(10**(digits // 2 - 1), 10**(digits // 2)); q = sympy.randprime(10**(digits // 2 - 1), 10**(digits // 2))
    n = p * q
    t0 = time.perf_counter(); g, size = factor_squares(n, B)
    print(f"n = {n} ({len(str(n))} digits): factor {g}, from a dependency of {size} relation(s), {time.perf_counter() - t0:.1f} s")

Most of the time goes into finding smooth values $x^2-n$ by trial division; the linear algebra is instantaneous at this size. The quadratic sieve replaces trial division by a sieve over the arithmetic progressions where $p\mid x^2-n$ (two residue classes modulo each $p$ of the base, given by the square roots of $n$ modulo $p$ from P10.2), which is what makes it practical for numbers of a hundred digits. Each dependency gives $X^2\equiv Y^2$, and by Proposition 10.20 at least half of them split $n$; in practice the first one almost always does.